# SPATIAL INTELLIGENCE - VISIBILITY GRAPH ANALYSIS

Compute a visibility graph from the prison floor plan and derive a VGA heatmap based on visual connectivity.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
BREP_PATH = os.path.join(BASE_DIR, "geometries", "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2
VGA_GRID_SIZE = 10

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan and create grids

We use a **coarse vertex grid** for visibility graph viewpoints and a **fine edge grid** for the analysis shell.

In [ ]:
# Load the cleaned floor plan
floor_plan = Topology.ByBREPPath(BREP_PATH)

# Bounding rectangle
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

# Coarse grid for VGA viewpoints
uRange1 = list(range(0, int(width) + VGA_GRID_SIZE, VGA_GRID_SIZE))
vRange1 = list(range(0, int(length) + VGA_GRID_SIZE, VGA_GRID_SIZE))
grid_viewpoints = Grid.VerticesByDistances(floor_plan, clip=True, uRange=uRange1, vRange=vRange1)

# Fine grid for analysis
uRange2 = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange2 = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid_edges = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange2, vRange=vRange2)

## 6. Slice the floor plan and derive the analysis graph

In [ ]:
shell = Topology.Slice(floor_plan, grid_edges)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell: {len(faces)} faces")

analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid_viewpoints)
print(f"Isovist viewpoints: {len(iso_verts)}")

## 7. Compute the visibility graph

`Graph.VisibilityGraph` connects pairs of viewpoints that can see each other within the floor plan. The holes in the face act as obstacles.

In [ ]:
import time

vga_verts = Topology.Vertices(grid_viewpoints)
print(f"Computing visibility graph with {len(vga_verts)} viewpoints...")

t0 = time.time()
vis_graph = Graph.VisibilityGraph(floor_plan, viewpointsA=vga_verts)
elapsed = time.time() - t0

vis_verts = Graph.Vertices(vis_graph)
vis_edges = Graph.Edges(vis_graph)
print(f"Visibility graph: {len(vis_verts)} vertices, {len(vis_edges)} edges ({elapsed:.1f}s)")

## 8. Show the visibility graph overlaid on the floor plan

In [ ]:
fig = Topology.Show(floor_plan, vis_graph,
              faceOpacity=0.3,
              faceColor=[210, 210, 250],
              vertexSize=4,
              vertexColor="red",
              edgeColor="yellow",
              edgeWidth=1,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "13_visibility_graph.png")

## 9. Compute visibility scores (degree centrality)

The visibility degree of each viewpoint is how many other viewpoints it can see. This gives a measure of visual connectivity — high-degree vertices are in areas with broad sightlines.

In [ ]:
new_verts = []
n_list = []
for v in vis_verts:
    degree = Graph.VertexDegree(vis_graph, v)
    n_list.append(degree)
    d = Dictionary.ByKeyValue("visibility", degree)
    v = Topology.SetDictionary(v, d)
    new_verts.append(v)

print(f"Visibility degree range: {min(n_list)} - {max(n_list)}")
print(f"Mean visibility degree: {sum(n_list) / len(n_list):.1f}")

## 10. Interpolate visibility to the full grid and color-map

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    v = Topology.SetDictionary(v, d)
print("Interpolation and color mapping complete.")

Transfer to the shell faces

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 11. Show visibility heatmap

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              showFigure=False, renderer=renderer)
save_fig(fig, "14_visibility_heatmap.png")